In [ ]:
import os
WORKSHOP_RESOURCE_GROUP = os.getenv("WORKSHOP_RESOURCE_GROUP", os.getenv("RESOURCE_GROUP_NAME", "rg-delete-me-01")).strip()
WORKSHOP_AUTH_MODE = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
os.environ["CHAT_DEPLOYMENT_NAME"] = ""
os.environ["CHAT_MODEL_NAME"] = ""
os.environ["EMBEDDING_DEPLOYMENT_NAME"] = ""
os.environ["EMBEDDING_MODEL_NAME"] = ""
os.environ["KB_MCP_ENDPOINT"] = ""
os.environ["RESOURCE_GROUP_NAME"] = WORKSHOP_RESOURCE_GROUP
os.environ["FOUNDRY_PROJECT_NAME"] = ""

In [ ]:
import os
import shlex
import subprocess

resource_group_name = os.getenv("WORKSHOP_RESOURCE_GROUP", os.getenv("RESOURCE_GROUP_NAME", "rg-delete-me-01")).strip()
auth_mode = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
foundry_project_name = os.getenv("FOUNDRY_PROJECT_NAME", "").strip()

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    resource_group_name,
    "--auth-mode",
    auth_mode,
]

if foundry_project_name:
    cmd.extend(["--foundry-project", foundry_project_name])

print(
    "Resolving workshop environment:",
    f"resource_group={resource_group_name}",
    f"project={foundry_project_name or '<auto>'}",
)

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

Resolving workshop environment: resource_group=rg-delete-me-01 project=proj-rg-delete-me-01-tyslc
Authentication mode: managed_identity
Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.


In [16]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

{
  "resource_group_name": "rg-delete-me-01",
  "location": "westus",
  "subscription_id": "7d8f212e-1960-4a5f-a3db-b77da9294085",
  "foundry_account_name": "hub-rg-delete-me-01-tyslc",
  "foundry_project_name": "proj-rg-delete-me-01-tyslc",
  "foundry_project_endpoint": "https://hub-rg-delete-me-01-tyslc.services.ai.azure.com/api/projects/hub-rg-delete-me-01-tyslc/proj-rg-delete-me-01-tyslc",
  "foundry_project_api_key": "",
  "search_service_name": "srch-wkshop-tyslc",
  "search_api_key": "",
  "storage_account_name": "stwkshoptyslc",
  "application_insights_name": "appi-wkshop-tyslc",
  "model_zone": "global"
}


# Workshop 3: Agents

This notebook mirrors docs/agents.md and creates the main workshop agents using code.

Agents created in this notebook:
- stats-for-coffee (Code Interpreter with CSV files)
- coffee-research-agent (retrieval-focused instruction set)

In [17]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity

In [18]:
import os
import subprocess
from pathlib import Path

from azure.ai.projects.models import AutoCodeInterpreterToolParam, CodeInterpreterTool, PromptAgentDefinition

from workshop_bootstrap import build_project_client

if not config.foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT or provide foundry_project_endpoint in CONFIG_OVERRIDES.")

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment.")

DATASET_GENERAL = Path("../../data/Coffee/CoffeeCSV/GeneralHealth/synthetic_mental_health_dataset.csv").resolve()
DATASET_LARGE = Path("../../data/Coffee/CoffeeCSV/mentalHealth/synthetic_coffee_health_10000.csv").resolve()

if not DATASET_GENERAL.exists() or not DATASET_LARGE.exists():
    raise FileNotFoundError("Required CSV files were not found in data/Coffee/CoffeeCSV.")

def normalize_project_name(raw_value: str) -> str:
    value = (raw_value or "").strip().strip("/")
    if not value:
        return ""
    if "/projects/" in value:
        value = value.split("/projects/", 1)[-1]
    return value.split("/")[-1]

if not config.foundry_account_name or not config.foundry_project_name:
    raise ValueError(
        "FOUNDRY_ACCOUNT_NAME and FOUNDRY_PROJECT_NAME are required. "
        "Re-run Cell 2 with the correct --resource-group and optional --foundry-project."
    )

normalized_project_name = normalize_project_name(config.foundry_project_name)
if normalized_project_name != config.foundry_project_name:
    print(
        "Normalizing project name:",
        f"{config.foundry_project_name} -> {normalized_project_name}",
    )
    config.foundry_project_name = normalized_project_name

if "/api/projects/" in config.foundry_project_endpoint:
    endpoint_prefix = config.foundry_project_endpoint.split("/api/projects/", 1)[0]
    normalized_endpoint = f"{endpoint_prefix}/api/projects/{config.foundry_project_name}"
    if normalized_endpoint != config.foundry_project_endpoint:
        print(
            "Normalizing endpoint project path:",
            f"{config.foundry_project_endpoint} -> {normalized_endpoint}",
        )
        config.foundry_project_endpoint = normalized_endpoint

project_probe_cmd = [
    "az",
    "cognitiveservices",
    "account",
    "project",
    "show",
    "--name",
    config.foundry_account_name,
    "--resource-group",
    config.resource_group_name,
    "--project-name",
    config.foundry_project_name,
    "--query",
    "name",
    "--output",
    "tsv",
]

probe = subprocess.run(project_probe_cmd, check=False, capture_output=True, text=True)
if probe.returncode != 0:
    stderr = (probe.stderr or "").strip()
    details = [
        "Foundry project validation failed before upload.",
        f"resource_group={config.resource_group_name}",
        f"account={config.foundry_account_name}",
        f"project={config.foundry_project_name}",
        f"endpoint={config.foundry_project_endpoint}",
        "Fix: re-run Cell 2 with the correct --resource-group and --foundry-project values.",
    ]

    list_projects_cmd = [
        "az",
        "cognitiveservices",
        "account",
        "project",
        "list",
        "--name",
        config.foundry_account_name,
        "--resource-group",
        config.resource_group_name,
        "--query",
        "[].name",
        "--output",
        "tsv",
    ]
    listed = subprocess.run(list_projects_cmd, check=False, capture_output=True, text=True)
    if listed.returncode == 0 and listed.stdout.strip():
        raw_names = [line.strip() for line in listed.stdout.splitlines() if line.strip()]
        normalized_names = sorted({normalize_project_name(name) for name in raw_names})
        if normalized_names:
            details.append("Available projects in this account: " + ", ".join(normalized_names))

    if stderr:
        details.append(f"Azure CLI error: {stderr}")
    raise RuntimeError("\n".join(details))

print(
    "Validated Foundry project:",
    f"{config.foundry_project_name} (account={config.foundry_account_name}, rg={config.resource_group_name})",
)

project = build_project_client(config)
openai = project.get_openai_client()

Normalizing endpoint project path: https://hub-rg-delete-me-01-tyslc.services.ai.azure.com/api/projects/hub-rg-delete-me-01-tyslc/proj-rg-delete-me-01-tyslc -> https://hub-rg-delete-me-01-tyslc.services.ai.azure.com/api/projects/proj-rg-delete-me-01-tyslc
Validated Foundry project: proj-rg-delete-me-01-tyslc (account=hub-rg-delete-me-01-tyslc, rg=rg-delete-me-01)


In [19]:
try:
    with DATASET_GENERAL.open("rb") as file_handle:
        general_file = openai.files.create(purpose="assistants", file=file_handle)

    with DATASET_LARGE.open("rb") as file_handle:
        large_file = openai.files.create(purpose="assistants", file=file_handle)
except Exception as exc:
    message = str(exc)
    if "The project does not exist" in message or "ResourceNotFound" in message:
        raise RuntimeError(
            "File upload failed because the configured Foundry project was not found.\n"
            f"Current endpoint: {config.foundry_project_endpoint}\n"
            f"Current account/project: {config.foundry_account_name}/{config.foundry_project_name}\n"
            "Re-run Cell 2 with the correct --resource-group and --foundry-project, then run Cells 3-7 again."
        ) from exc
    raise

print("Uploaded files:")
print("- general:", general_file.id)
print("- large:  ", large_file.id)

Uploaded files:
- general: assistant-Q1xUaZYpJdem4Np3eYfua2
- large:   assistant-TxZRnrNzGccnN6EKrqoJQy


In [20]:
STATS_AGENT_NAME = "stats-for-coffee"
RESEARCH_AGENT_NAME = "coffee-research-agent"

STATS_AGENT_INSTRUCTIONS = """
You are a rigorous data-analysis agent for a scientific workshop in Microsoft Foundry.

Primary mission:
- Analyze two workshop CSV files using Code Interpreter.
- Explain findings in plain English and show the exact calculation logic.
- Generate charts when they help understanding.
- Be explicit about uncertainty, synthetic-data limitations, and non-causal interpretations.

Behavioral rules:
- Always inspect schema first before calculating.
- Use Python for numeric, statistical, plotting, grouping, filtering, and transformation tasks.
- Do not claim causation from correlation.
- If asked to compare both datasets, do thematic comparison unless a shared key is provided.

Preferred output format:
1. Question
2. Datasets and columns
3. Method
4. Results
5. Interpretation
6. Caveats
"""

RESEARCH_AGENT_INSTRUCTIONS = """
You are Coffee Research Agent, an academic research assistant.

Mission:
Provide evidence-based, citation-rich answers about coffee and health with clear uncertainty reporting.

Grounding rules:
- Prioritize scientific health evidence.
- Treat CSV-based observations as exploratory and non-clinical.
- Do not invent facts, statistics, or citations.
- Do not infer causation from correlation without explicit evidence.
"""

stats_agent = project.agents.create_version(
    agent_name=STATS_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=STATS_AGENT_INSTRUCTIONS,
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(
                    file_ids=[general_file.id, large_file.id]
                )
            )
        ],
    ),
    description="Workshop statistical analysis agent.",
)

research_agent = project.agents.create_version(
    agent_name=RESEARCH_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=RESEARCH_AGENT_INSTRUCTIONS,
        tools=[],
    ),
    description="Workshop evidence synthesis agent.",
)

print("Created agent versions:")
print("- stats agent:   ", stats_agent.name, stats_agent.version)
print("- research agent:", research_agent.name, research_agent.version)

Created agent versions:
- stats agent:    stats-for-coffee 1
- research agent: coffee-research-agent 1


In [21]:
conversation = openai.conversations.create()

def ask_agent(agent_name: str, prompt: str):
    return openai.responses.create(
        conversation=conversation.id,
        input=prompt,
        extra_body={
            "agent_reference": {
                "name": agent_name,
                "type": "agent_reference",
            }
        },
    )

prompt_text = "Start by inspecting both datasets and provide schema, missingness, and quality issues."
stats_response = ask_agent(STATS_AGENT_NAME, prompt_text)
stats_response

Response(id='resp_2ec5f636532d9c7b006a4d26173fb88190b4b3a3ece4ff0d0c', created_at=1783440919.0, error=None, incomplete_details=None, instructions='\nYou are a rigorous data-analysis agent for a scientific workshop in Microsoft Foundry.\n\nPrimary mission:\n- Analyze two workshop CSV files using Code Interpreter.\n- Explain findings in plain English and show the exact calculation logic.\n- Generate charts when they help understanding.\n- Be explicit about uncertainty, synthetic-data limitations, and non-causal interpretations.\n\nBehavioral rules:\n- Always inspect schema first before calculating.\n- Use Python for numeric, statistical, plotting, grouping, filtering, and transformation tasks.\n- Do not claim causation from correlation.\n- If asked to compare both datasets, do thematic comparison unless a shared key is provided.\n\nPreferred output format:\n1. Question\n2. Datasets and columns\n3. Method\n4. Results\n5. Interpretation\n6. Caveats\n', metadata={}, model='gpt-4-1', object=

## Monitoring

Use Application Insights from baseline deployment for agent monitoring in Foundry portal.

## Next
Continue with 05_multi_agent_workshop.ipynb for sequential multi-agent orchestration.